In [4]:
using Pkg
Pkg.activate("./")
Pkg.develop(path="../../")
ENV["TAMBOSIM_PATH"] = realpath("../../")

  Activating project at `~/research/TAMBO-MC/notebooks/create_geometry`
   Resolving package versions...
  No Changes to `~/research/TAMBO-MC/notebooks/create_geometry/Project.toml`
  No Changes to `~/research/TAMBO-MC/notebooks/create_geometry/Manifest.toml`


"/Users/jlazar/research/TAMBO-MC"

In [5]:
using CairoMakie
using Distributions
using GMT
using HDF5
using LinearAlgebra
using Makie
using Tambo

# Make triangulation of the Earth
## Sample points of different densities

In reality, we will need many, many more points than this, but sampling that densely makes visualization very hard.
This suffices for now.
We have also grossly exaggrated the distance scales for the same reason.

In [6]:
rearth = 6_378_000 # m
θtrans1, θtrans2 = 250_000 / rearth, 1_000_000 / rearth
N1, N2, N3 = 1_000, 1_000, 1_000

sample_points = zeros((N1 + N2 + N3, 2))

for idx in 1:N1
    sinlat = rand(Uniform(cos(θtrans1), 1))
    long = 2π * rand()
    sample_points[idx, :] = [long, asin(sinlat)]
end
    
for idx in 1:N2
    sinlat = rand(Uniform(cos(θtrans2), cos(θtrans1)))
    long = 2π * rand()
    sample_points[idx + N1, :] = [long, asin(sinlat)]
end
    
for idx in 1:N3
    sinlat = rand(Uniform(-1, cos(θtrans2)))
    long = 2π * rand()
    sample_points[idx + N1 + N2, :] = [long, asin(sinlat)]
end     

## Run the triangulation and display it

### Please note that the triangles that come out of this are all oriented away from the center
This is to say that `dot(centroid(tri), cross(tri.v1 - tri.v2, tri.v1 - tri.v3)) > 0`.
This is very important as we will rely on this to detect whether a point is inside of outside the mesh in the future.
If you elect to do triangulation by some other means, please ensure that this convention holds.

In [ ]:
@time triangulation = sphtriangulate(rad2deg.(sample_points), area=true, unit=:km)
m = imshow(triangulation, region=:global, proj=:guess, frame=:afg, pen=0.002)
display(m);

# To get any reasonable spatial resolution, we need to change the parameters
We will make the near and middle regions smaller, and increase the number of points that we consider

## First, let's do a quick study to understand the scaling

In [7]:
function spherical_triangle_area(p1, p2, p3, radius=1.0)
    """
    Calculate area of spherical triangle using spherical excess
    
    Args:
        p1, p2, p3: Points as (longitude, latitude) in degrees
        radius: Sphere radius (default: unit sphere)
    
    Returns:
        Area on sphere surface
    """
    # Convert to radians
    lon1, lat1 = deg2rad.(p1)
    lon2, lat2 = deg2rad.(p2)
    lon3, lat3 = deg2rad.(p3)
    
    # Calculate side lengths (angular distances) using spherical law of cosines
    a = acos(sin(lat2) * sin(lat3) + cos(lat2) * cos(lat3) * cos(lon3 - lon2))
    b = acos(sin(lat3) * sin(lat1) + cos(lat3) * cos(lat1) * cos(lon1 - lon3))
    c = acos(sin(lat1) * sin(lat2) + cos(lat1) * cos(lat2) * cos(lon2 - lon1))
    
    # Calculate spherical excess using L'Huilier's formula
    s = (a + b + c) / 2
    a = tan(s/2) * tan((s-a)/2) * tan((s-b)/2) * tan((s-c)/2)
    E = 4 * atan(sqrt(max(a, 0)))
    
    return radius^2 * E
end

spherical_triangle_area (generic function with 2 methods)

In [8]:
function sample_points_on_sphere(Ns::Tuple{Int, Int, Int}, θs::Tuple{Float64, Float64})
    N1, N2, N3 = Ns
    θtrans1, θtrans2 = θs
    
    sample_points = zeros((N1 + N2 + N3, 2))

    for idx in 1:N1
        sinlat = rand(Uniform(cos(θtrans1), 1))
        long = 2π * rand()
        sample_points[idx, :] = [long, asin(sinlat)]
    end

    for idx in 1:N2
        sinlat = rand(Uniform(cos(θtrans2), cos(θtrans1)))
        long = 2π * rand()
        sample_points[idx + N1, :] = [long, asin(sinlat)]
    end

    for idx in 1:N3
        sinlat = rand(Uniform(-1, cos(θtrans2)))
        long = 2π * rand()
        sample_points[idx + N1 + N2, :] = [long, asin(sinlat)]
    end
    return sample_points
end

sample_points_on_sphere (generic function with 1 method)

In [9]:
function calc_centroid(triangle::GMTdataset{Float64, 2})
    centroid = zeros(3)
    for idx in 1:3
        x = Tambo.longlat_to_cart(deg2rad.(triangle[idx, :])...)
        centroid .+= x
    end
    return centroid ./ 3
end

calc_centroid (generic function with 1 method)

In [10]:
function compute_triangulaiton_areas(
    triangulation::Vector{GMTdataset{Float64, 2}},
    θs::Tuple{Float64, Float64}
)
    θtrans1, θtrans2 = θs
    areas1, areas2, areas3 = [], [], []
    for triangle in triangulation
        centroid = calc_centroid(triangle)
        θ = acos(dot([0, 0, 1], centroid) / norm(centroid))
        p1 = triangle[1, :]
        p2 = triangle[2, :]
        p3 = triangle[3, :]
        a = spherical_triangle_area(p1, p2, p3, rearth)
        if θ < θtrans1
            push!(areas1, a)
        elseif θ < θtrans2
            push!(areas2, a)
        else
            push!(areas3, a)
        end
    end 
    return areas1, areas2, areas3
end

compute_triangulaiton_areas (generic function with 1 method)

## Warning: running this will take a couple minutes.
You can get the gist by uncommenting the fifth line, which will take <10 seconds

In [ ]:
θs = 10_000 / rearth, 50_000 / rearth

ns = [1_000, 3_000, 10_000, 30_000, 100_000]
# ns = [1_000, 3_000, 10_000, 30_000]
outs = zeros((length(ns), 3, 3))

for (idx, N) in enumerate(ns)
    Ns = N, N, N

    sample_points = sample_points_on_sphere(Ns, θs)
    
    triangulation = sphtriangulate(rad2deg.(sample_points), area=true, unit=:km)
    areas1, areas2, areas3 = compute_triangulaiton_areas(triangulation, θs)
    
    outs[idx, 1, :] = sqrt.(quantile(areas1, [0.16, 0.5, 0.84]))
    outs[idx, 2, :] = sqrt.(quantile(areas2, [0.16, 0.5, 0.84]))
    outs[idx, 3, :] = sqrt.(quantile(areas3, [0.16, 0.5, 0.84]))
end

fig = Figure()
ax = Axis(
    fig[1,1],
    xscale=log10,
    yscale=log10,
    xlabel="Npoint / region",
    ylabel="√(Area) [m]",
)

Makie.lines!(ax, ns, outs[:, 3, 2], label="Far", color="blue")
Makie.lines!(ax, ns, outs[:, 2, 2], label="Middle", color="green")
Makie.lines!(ax, ns, outs[:, 1, 2], label="Near", color="red")

Makie.fill_between!(ax, ns, outs[:, 1, 1], outs[:, 1, 3], alpha=0.3, color="red")
Makie.fill_between!(ax, ns, outs[:, 2, 1], outs[:, 2, 3], alpha=0.3, color="green")
Makie.fill_between!(ax, ns, outs[:, 3, 1], outs[:, 3, 3], alpha=0.3, color="blue")

axislegend(ax)

fig

# If we want ~10m resolution, we need to do 1,000,000 points per region
This should give us resolution on the order of 50m in the middle region and 10-20km everywhere else. 
## Warning: This will likely take a few hours, but it only have to happen once.

In [37]:
θtrans1, θtrans2 = 10_000 / rearth, 50_000 / rearth
N1, N2, N3 = 10_000, 10_000, 10_000

sample_points = zeros((N1 + N2 + N3, 2))

for idx in 1:N1
    sinlat = rand(Uniform(cos(θtrans1), 1))
    long = 2π * rand()
    sample_points[idx, :] = [long, asin(sinlat)]
end
    
for idx in 1:N2
    sinlat = rand(Uniform(cos(θtrans2), cos(θtrans1)))
    long = 2π * rand()
    sample_points[idx + N1, :] = [long, asin(sinlat)]
end
    
for idx in 1:N3
    sinlat = rand(Uniform(-1, cos(θtrans2)))
    long = 2π * rand()
    sample_points[idx + N1 + N2, :] = [long, asin(sinlat)]
end

@time triangulation = sphtriangulate(rad2deg.(sample_points), area=true, unit=:km)

  0.905868 seconds (1.14 M allocations: 54.633 MiB)


Vector{GMTdataset{Float64, 2}} with 59996 segments
Showing first segment. To see other segments just type its element number. e.g. D[2]

Comment:	["sphtriangulate Delaunay output via STRPACK.  Areas in km^2."]
BoundingBox: [169.88350124216112, 170.4702039912073, 89.94416994158426, 89.94557280738563]
Global BoundingBox: [0.004290521220866594, 359.9956126535662, -89.21236379305145, 89.99895867786498]
Header:	Triangle: 0 0-6014-6908 Area: 0.00289707 -Z0

3×2 GMTdataset{Float64, 2}
 Row │   col.1    col.2
─────┼──────────────────
   1 │ 169.884  89.9456
   2 │ 170.47   89.9442
   3 │ 170.412  89.9451

# Save it to a file
Now that we have this beautiful triangulation, we should save it to a file so that we can reuse it and don't have to run a multi-hour calculation again.

In [ ]:
vertices, faces = Tambo.triangles_to_mesh(triangulation)

# I forget why this has to happen....
tmp = zeros((length(vertices), 2))
for idx in 1:length(vertices)
    tmp[idx, :] = vertices[idx]
end
vertices = tmp

tmp = zeros(Int64, (length(faces), 3))
for idx in 1:length(faces)
    tmp[idx, :] .= faces[idx]
end
faces = tmp

# h5open("triangulation.h5", "w") do h5f
#     group = create_group(h5f, "base_triangulation")
#     group["vertices"] = vertices
#     group["faces"] = faces
# end;

In [11]:
θtrans1, θtrans2 = 2_000 / rearth, 20_000 / rearth
N1, N2, N3 = 40_000, 40_000, 40_000

sample_points = zeros((N1 + N2 + N3, 2))

for idx in 1:N1
    sinlat = rand(Uniform(cos(θtrans1), 1))
    long = 2π * rand()
    sample_points[idx, :] = [long, asin(sinlat)]
end
    
for idx in 1:N2
    sinlat = rand(Uniform(cos(θtrans2), cos(θtrans1)))
    long = 2π * rand()
    sample_points[idx + N1, :] = [long, asin(sinlat)]
end
    
for idx in 1:N3
    sinlat = rand(Uniform(-1, cos(θtrans2)))
    long = 2π * rand()
    sample_points[idx + N1 + N2, :] = [long, asin(sinlat)]
end

@time triangulation = sphtriangulate(rad2deg.(sample_points), area=true, unit=:km)

 15.423092 seconds (5.57 M allocations: 272.224 MiB, 2.26% gc time, 2.25% compilation time)
Vector{GMTdataset{Float64, 2}} with 239996 segments
Showing first segment. To see other segments just type its element number. e.g. D[2]

Comment:	["sphtriangulate Delaunay output via STRPACK.  Areas in km^2."]
BoundingBox: 

[79.50291424700023, 80.39433037523, 89.99201778456887, 89.99224148833044]
Global BoundingBox: [0.0004577549409123804, 359.9981258498351, -89.25068447407966, 89.9998242905626]
Header:	Triangle: 0 0-18699-9062 Area: 5.79519e-05 -Z0

3×2 GMTdataset{Float64, 2}
 Row │   col.1    col.2
─────┼──────────────────
   1 │ 79.945   89.9921
   2 │ 80.3943  89.992
   3 │ 79.5029  89.9922

In [17]:
triangulation[2][1,:]

2-element Vector{Float64}:
 79.94499503181521
 89.99205647897722

In [47]:
h5open("triangulation.h5") do h5f
    @show h5f["base_triangulation_1000/faces"][1:10, :]
end

(h5f["base_triangulation_1000/faces"])[1:10, :] = [1 2 3; 1 3 4; 1 4 5; 1 5 2; 6 7 8; 6 8 9; 6 9 10; 6 10 11; 6 11 12; 6 12 7]


10×3 Matrix{Int64}:
 1   2   3
 1   3   4
 1   4   5
 1   5   2
 6   7   8
 6   8   9
 6   9  10
 6  10  11
 6  11  12
 6  12   7

In [31]:
typeof(findfirst(1 .== [2,3,4]))

Nothing

In [39]:
faces

59996-element Vector{Vector{Int64}}:
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 ⋮
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]
 [1, 1, 1]

In [46]:
faces

59996-element Vector{Vector{Int64}}:
 [1, 2, 3]
 [4, 3, 5]
 [6, 5, 7]
 [8, 7, 9]
 [10, 9, 11]
 [12, 11, 2]
 [13, 14, 15]
 [13, 15, 16]
 [13, 16, 17]
 [13, 17, 18]
 [13, 18, 14]
 [19, 20, 21]
 [19, 21, 22]
 ⋮
 [23316, 21898, 27196]
 [24864, 28054, 22061]
 [29334, 29866, 29765]
 [29765, 29866, 29718]
 [28880, 23924, 21808]
 [27729, 28218, 29960]
 [29624, 29960, 28218]
 [29624, 28218, 28217]
 [27616, 27615, 30003]
 [29371, 26426, 26425]
 [29959, 22989, 27934]
 [22461, 28573, 25156]

In [49]:
using ProgressMeter

In [65]:
for i in eachrow(triangulation[1])
    @show i 
end

i = [169.88350124216112, 89.94557280738563]
i = [170.4702039912073, 89.94416994158426]
i = [170.41158932536348, 89.94514480915318]


In [68]:
function triangles_to_mesh_geographic(triangles; 
                                      tol::Float64=1e-8)
    # triangles: list of triangles, each triangle is a vector of 3 points
    # each point is [longitude, latitude] or [longitude, latitude, elevation?]
    
    vertices = Vector{Vector{Float64}}()
    faces = Vector{Tuple{Int,Int,Int}}()
    
    # Dictionary with normalized coordinates for matching
    vertex_to_idx = Dict{Tuple{Float64,Float64}, Int}()
    
    for tri in triangles
        idxs = zeros(Int, 3)
        for (i, v) in enumerate(eachrow(tri))
#             @show v
            # Extract lon, lat (and ignore elevation if present)
            lon, lat = v[1], v[2]
            
            # Normalize longitude to [-180, 180] for consistent matching
            lon_norm = mod(lon + 180, 360) - 180
            
            # Round to tolerance for matching
            lon_key = round(lon_norm / tol) * tol
            lat_key = round(lat / tol) * tol
            key = (lon_key, lat_key)
            
            if haskey(vertex_to_idx, key)
                idxs[i] = vertex_to_idx[key]
            else
                # Store original coordinates, not rounded
                push!(vertices, v)
                new_idx = length(vertices)
                vertex_to_idx[key] = new_idx
                idxs[i] = new_idx
            end
        end
        push!(faces, (idxs[1], idxs[2], idxs[3]))
    end
    
    return vertices, faces
end

triangles_to_mesh_geographic (generic function with 2 methods)

In [69]:
triangles_to_mesh_geographic(triangulation)

([[169.88350124216112, 89.94557280738563], [170.4702039912073, 89.94416994158426], [170.41158932536348, 89.94514480915318], [170.2897801843985, 89.94596169168419], [168.27061973376277, 89.94688779098928], [168.12499058674928, 89.94633238220828], [168.8103395598106, 89.9433861204529], [261.43292703790064, 89.91873046285828], [261.3968187394092, 89.9186176781951], [261.86616064155623, 89.91855454921402]  …  [68.53984989155703, -11.476006547050082], [93.3848705031669, 21.55609229939684], [156.68685364194184, 15.604383589616525], [121.43773167697843, -55.94407604484408], [191.76887490099196, -69.45170871452204], [93.6237099186317, 55.43067469106254], [111.87397756458948, 2.677434362194731], [330.4971964653076, -64.98754319204164], [255.8663437117316, 53.9642514137477], [255.14523762498268, 53.716565620732844]], [(1, 2, 3), (1, 3, 4), (1, 4, 5), (1, 5, 6), (1, 6, 7), (1, 7, 2), (8, 9, 10), (8, 10, 11), (8, 11, 12), (8, 12, 13)  …  (29329, 29861, 29760), (29760, 29861, 29713), (28875, 23919,